## **temporal.py**

This is where the system decides.

Everything before this produced observations. Detection found heads,
association matched them to riders, the manager collected them. Nobody has yet
said "this rider was not wearing a helmet."

The reason that's a separate step is that a single frame can't be trusted. A
rider passes behind a pole, the image blurs on a bump, the sun catches a dark
helmet at the wrong angle. Any one frame might be wrong. Two small functions. One summarises a rider's observations; the other decides
whether that summary is strong enough to act on.

## **Setup**

In [1]:
!git clone -q https://github.com/shreyamali17/helmet-violation-detection.git
%cd helmet-violation-detection/project/pipeline
!pip install -q ultralytics scipy

/content/helmet-violation-detection/project/pipeline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.9 MB/s eta 0:00:00


## **What arrives here**

The `observations` dictionary accumulates during the pipeline loop. Each rider
maps to a list of `(status, model_confidence)` tuples : one entry per frame
where that rider was matched to a head detection.

```python
observations[7] = [
    ("no_helmet", 0.91),
    ("no_helmet", 0.88),
    ("helmet",    0.62),   # one frame disagreed
    ("no_helmet", 0.93),
]
```

Note what's in the second slot: the **model's** confidence in that individual
detection. This becomes important shortly.

## **Smoothing**

Take the majority status, and report a confidence for it.

In [2]:
from collections import Counter
from config import MIN_OBSERVATIONS, MIN_CONFIDENCE


def smooth_helmet_status(observations, rider_id, min_observations=None):
    min_observations = min_observations if min_observations is not None else MIN_OBSERVATIONS
    votes = observations.get(rider_id, [])

    # not enough evidence to say anything at all
    if len(votes) < min_observations:
        return None

    # majority vote on the status
    statuses = [v[0] for v in votes]
    status, _ = Counter(statuses).most_common(1)[0]

    # average the MODEL's confidence, over agreeing frames only
    agreeing_confs = [v[1] for v in votes if v[0] == status and v[1] is not None]
    avg_confidence = sum(agreeing_confs) / len(agreeing_confs) if agreeing_confs else None

    return status, avg_confidence

### **Returning `None` is a real answer**

```python
if len(votes) < min_observations:
    return None
```

There are three possible outcomes here, not two: helmet, no helmet, and *not
enough evidence*. That third one is a feature.

A rider glimpsed twice at the far edge of the frame produces `None`. Not a
guess, not a default to "compliant" an explicit refusal to decide. Most of
the riders in any given video end up here, and that's the system working as
intended.

### **Which confidence is being averaged**

This line deserves attention, because it's easy to misread:

```python
agreeing_confs = [v[1] for v in votes if v[0] == status and v[1] is not None]
```

It averages **YOLO's own confidence scores**, taken only over the frames that
agree with the majority. It is *not* the fraction of frames that agreed.

Those are different numbers and they answer different questions:

| | Question it answers |
|---|---|
| Vote agreement | How consistent was the rider across frames? |
| Average model confidence | How sure was the detector when it agreed? |

The code uses the second. A consequence follows from that, and it's worth
seeing rather than describing.

In [3]:
# two riders, both majority no_helmet, both with 3 observations
unanimous = [("no_helmet", 0.90), ("no_helmet", 0.88), ("no_helmet", 0.92)]
split     = [("no_helmet", 0.90), ("helmet", 0.85), ("no_helmet", 0.92)]

obs = {1: unanimous, 2: split}

for rid, votes in obs.items():
    statuses = [v[0] for v in votes]
    agreement = statuses.count(Counter(statuses).most_common(1)[0][0]) / len(votes)
    result = smooth_helmet_status(obs, rid)
    print(f"rider {rid}: {statuses}")
    print(f"   vote agreement:  {agreement:.0%}")
    print(f"   smoothed result: {result[0]}, confidence {result[1]:.2f}")
    print()

rider 1: ['no_helmet', 'no_helmet', 'no_helmet']
   vote agreement:  100%
   smoothed result: no_helmet, confidence 0.90

rider 2: ['no_helmet', 'helmet', 'no_helmet']
   vote agreement:  67%
   smoothed result: no_helmet, confidence 0.91



Rider 1 agreed with itself every frame. Rider 2 disagreed a third of the time.
Both come out with roughly the same reported confidence, because the frames
that disagreed are simply excluded from the average.

The dissent is invisible in the output. Whether that's the right behaviour
depends on what you want the number to mean but it's worth knowing that the
reported confidence says nothing about consistency.

## **Deciding**

The second function applies the threshold and returns a verdict.

In [4]:
def decide_violation(smoothed_result, min_confidence=None):
    min_confidence = min_confidence if min_confidence is not None else MIN_CONFIDENCE

    if smoothed_result is None:
        return None                      # not enough observations

    status, confidence = smoothed_result
    if confidence is None or confidence < min_confidence:
        return None                      # not confident enough

    return status == "no_helmet"         # True or False

### **Three-valued logic**

This function returns `True`, `False`, or `None`, and all three mean something
distinct:

- **`True`** : confirmed violation
- **`False`** : confirmed wearing a helmet
- **`None`** : the system declines to say

That's why the code elsewhere is careful to write `if inst.is_violation is
True` rather than `if inst.is_violation`. In Python, `None` and `False` are
both falsy, so a plain truth test collapses "no violation" and "don't know"
into the same branch.

For a system that might generate a traffic ticket, that distinction is the
whole point.

### **Why the split into two functions**

`smooth_helmet_status` answers "what does the evidence say?" and
`decide_violation` answers "is that enough?"

Keeping them separate means you can inspect the evidence without committing to
a verdict useful for debugging, and for showing a human operator *why* a
violation was flagged rather than just that it was. It also means the two failure modes stay distinguishable. Too few observations
and insufficient confidence both return `None` from `decide_violation`, but
`smooth_helmet_status` tells you which one happened.

## **Decision Table**

Running a range of cases through both functions.

In [5]:
cases = {
    "seen once":            [("no_helmet", 0.95)],
    "seen twice":           [("no_helmet", 0.95), ("no_helmet", 0.93)],
    "3x confident":         [("no_helmet", 0.91), ("no_helmet", 0.88), ("no_helmet", 0.94)],
    "3x low confidence":    [("no_helmet", 0.55), ("no_helmet", 0.61), ("no_helmet", 0.58)],
    "helmet, confident":    [("helmet", 0.89), ("helmet", 0.92), ("helmet", 0.87)],
    "split, high conf":     [("no_helmet", 0.93), ("helmet", 0.80), ("no_helmet", 0.91)],
    "borderline 0.70":      [("no_helmet", 0.70), ("no_helmet", 0.70), ("no_helmet", 0.70)],
}

print(f"{'case':<22}{'smoothed':<28}{'violation'}")
print("-" * 66)
for name, votes in cases.items():
    o = {1: votes}
    s = smooth_helmet_status(o, 1)
    v = decide_violation(s)
    s_str = "None (too few obs)" if s is None else f"{s[0]}, conf {s[1]:.2f}"
    v_str = {True: "TRUE - violation", False: "False - compliant", None: "None - unknown"}[v]
    print(f"{name:<22}{s_str:<28}{v_str}")

case                  smoothed                    violation
------------------------------------------------------------------
seen once             None (too few obs)          None - unknown
seen twice            None (too few obs)          None - unknown
3x confident          no_helmet, conf 0.91        TRUE - violation
3x low confidence     no_helmet, conf 0.58        None - unknown
helmet, confident     helmet, conf 0.89           False - compliant
split, high conf      no_helmet, conf 0.92        TRUE - violation
borderline 0.70       no_helmet, conf 0.70        None - unknown


Three of these are worth dwelling on.

**"seen twice"** has a detection at 0.95 confidence and still returns nothing.
The observation count gate runs first and doesn't care how confident any single
frame was.

**"3x low confidence"** clears the count but fails the threshold. The detector
saw the same thing three times and wasn't sure any of them.

**"split, high conf"** is confirmed as a violation despite one frame in three
disagreeing the consequence of averaging model confidence rather than vote
agreement.

## **On real video**

Run the first stages and see what the decision layer does with actual
observations.

In [6]:
from ultralytics import YOLO
import config
from detection import get_frame_detections
from association import associate_heads_to_riders

model = YOLO(config.MODEL_PATH)

results = model.track(
    source=config.VIDEO_PATH,
    classes=list(config.CLASS_NAMES.keys()),
    tracker=config.TRACKER,
    persist=True, stream=True, verbose=False,
)

observations = {}
FRAME_LIMIT = 300

for i, r in enumerate(results):
    if i >= FRAME_LIMIT:
        break
    det, confs = get_frame_detections(r)
    matches = associate_heads_to_riders(
        det["rider"], det["helmet"], det["no_helmet"],
        helmet_confs=confs["helmet"], no_helmet_confs=confs["no_helmet"],
    )
    for rider_id, (status, model_conf) in matches.items():
        observations.setdefault(rider_id, []).append((status, model_conf))

print(f"riders with at least one head observation: {len(observations)}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 251ms
Prepared 1 package in 62ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

riders with at least one head observation: 22


In [7]:
outcomes = {"violation": [], "compliant": [], "too few obs": [], "not confident": []}

for rider_id in observations:
    s = smooth_helmet_status(observations, rider_id)
    v = decide_violation(s)
    n = len(observations[rider_id])

    if s is None:
        outcomes["too few obs"].append(rider_id)
    elif v is None:
        outcomes["not confident"].append((rider_id, n, s[1]))
    elif v:
        outcomes["violation"].append((rider_id, n, s[1]))
    else:
        outcomes["compliant"].append((rider_id, n, s[1]))

for label, items in outcomes.items():
    print(f"{label:<16} {len(items)}")

print("\nconfirmed violations:")
for rid, n, conf in outcomes["violation"]:
    print(f"  rider {rid:<6} {n:>3} observations, avg confidence {conf:.2f}")

violation        3
compliant        3
too few obs      4
not confident    12

confirmed violations:
  rider 1       63 observations, avg confidence 0.76
  rider 6        6 observations, avg confidence 0.70
  rider 420      5 observations, avg confidence 0.73


The "too few obs" group is usually the largest. That's the system correctly
declining to judge riders it barely saw.

Look at one rider's raw observation list to see what smoothing actually
operated on.

In [8]:
if outcomes["violation"]:
    rid = outcomes["violation"][0][0]
    votes = observations[rid]

    print(f"rider {rid} - {len(votes)} observations\n")
    for k, (status, conf) in enumerate(votes):
        bar = "#" * int((conf or 0) * 20)
        print(f"  {k+1:>3}. {status:<11} {conf:.2f}  {bar}")

    statuses = [v[0] for v in votes]
    top = Counter(statuses).most_common(1)[0][0]
    print(f"\n  majority:        {top} ({statuses.count(top)}/{len(statuses)} frames)")
    print(f"  vote agreement:  {statuses.count(top)/len(statuses):.0%}")
    print(f"  reported result: {smooth_helmet_status(observations, rid)}")
else:
    print("no violations in this frame range - try raising FRAME_LIMIT")

rider 1 - 63 observations

    1. no_helmet   0.82  ################
    2. no_helmet   0.81  ################
    3. no_helmet   0.74  ##############
    4. no_helmet   0.75  ##############
    5. no_helmet   0.64  ############
    6. no_helmet   0.61  ############
    7. no_helmet   0.64  ############
    8. no_helmet   0.77  ###############
    9. no_helmet   0.81  ################
   10. no_helmet   0.79  ###############
   11. no_helmet   0.76  ###############
   12. no_helmet   0.73  ##############
   13. no_helmet   0.68  #############
   14. no_helmet   0.70  ##############
   15. no_helmet   0.67  #############
   16. no_helmet   0.73  ##############
   17. no_helmet   0.74  ##############
   18. no_helmet   0.71  ##############
   19. no_helmet   0.78  ###############
   20. no_helmet   0.78  ###############
   21. no_helmet   0.76  ###############
   22. no_helmet   0.74  ##############
   23. no_helmet   0.73  ##############
   24. no_helmet   0.75  ##############
   25. no

## **Try It**

**Drop `MIN_OBSERVATIONS` to 1.** Every rider seen once becomes eligible.

```python
smooth_helmet_status(observations, rider_id, min_observations=1)
```

Count the extra violations. Do they hold up when you look at the frames?

**Raise `MIN_CONFIDENCE` to 0.9.** How many confirmed violations survive? Being
stricter means fewer false accusations and more missed violations there's no
setting that avoids both.

**Add a vote-agreement check.** Write a version of `decide_violation` that also
requires, say, 80% of frames to agree on the status. How many of the current
violations does it reject?

That last one is the interesting exercise. It's a small change with a real
effect on which riders get flagged, and it forces the question of what
"confident" should mean in a system that issues tickets.

## **Next**

At this point every rider has a verdict. What's left is turning that into
something a person can look at boxes drawn on frames, evidence snapshots
saved to disk. That's `visualization.py`.